In [7]:
# UNDERSTANDING THE DATA

import pandas as pd
brands = pd.read_csv('../data/raw/nykaa_brands.csv')
brands.info()
brands.describe()
brands.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3663 entries, 0 to 3662
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   brand_name         3663 non-null   object 
 1   product_id         3663 non-null   int64  
 2   image_url          3663 non-null   object 
 3   in_stock           3363 non-null   object 
 4   mrp                3663 non-null   int64  
 5   price              3663 non-null   int64  
 6   product_title      3663 non-null   object 
 7   rating             3663 non-null   float64
 8   rating_count       3663 non-null   int64  
 9   tags               305 non-null    object 
 10  product_url        3663 non-null   object 
 11  listing_page_name  3663 non-null   object 
 12  listing_url        3663 non-null   object 
 13  listing_page_no    3663 non-null   int64  
dtypes: float64(1), int64(5), object(8)
memory usage: 400.8+ KB


,brand_name,product_id,image_url,in_stock,mrp,price,product_title,rating,rating_count,tags,product_url,listing_page_name,listing_url,listing_page_no
0,Herbal Essences,2659739,https://images-static.nykaa.com/media/catalog/...,True,1250,750,Herbal Essences Argan Oil Of Moroccan Shampoo ...,4.4,1008,"FEATURED, BESTSELLER",https://www.nykaa.com/herbal-essences-argan-oi...,Herbal Essences,https://www.nykaa.com/brands/herbal-essences/c...,1
1,Herbal Essences,1290145,https://images-static.nykaa.com/media/catalog/...,True,1575,1181,Herbal Essences Aloe & Bamboo Shampoo + Condit...,4.4,1034,FEATURED,https://www.nykaa.com/herbal-essences-potent-a...,Herbal Essences,https://www.nykaa.com/brands/herbal-essences/c...,1
2,Herbal Essences,456559,https://images-static.nykaa.com/media/catalog/...,True,1250,688,Herbal Essences Argan Oil Shampoo & Conditione...,4.3,10879,"FEATURED, BESTSELLER",https://www.nykaa.com/herbal-essences-argan-oi...,Herbal Essences,https://www.nykaa.com/brands/herbal-essences/c...,1
3,Herbal Essences,3753166,https://images-static.nykaa.com/media/catalog/...,True,1575,945,Herbal Essences Aloe & Bamboo Shampoo + Condit...,4.3,65,FEATURED,https://www.nykaa.com/herbal-essences-soha-alo...,Herbal Essences,https://www.nykaa.com/brands/herbal-essences/c...,1
4,Herbal Essences,5360837,https://images-static.nykaa.com/media/catalog/...,True,600,390,Herbal Essences Argan Oil Of Morocco Shampoo -...,4.3,8769,"FEATURED, BESTSELLER",https://www.nykaa.com/herbal-essences-argan-oi...,Herbal Essences,https://www.nykaa.com/brands/herbal-essences/c...,1


In [12]:
# DATA CLEANING

# Standardizing brand name casing
brands['brand_name'] = brands['brand_name'].str.strip().str.title()
# Filling missing in_stock values 
print(brands['in_stock'].isnull().sum()) # confirming the count
brands['in_stock'] = brands['in_stock'].fillna('Unknown')
# Turning blank tags into an explicit category
brands['tags'] = brands['tags'].fillna('Untagged')
# Creating a discount percentage column from mrp and price
brands['discount_pct'] = ((brands['mrp'] - brands['price']) / brands['mrp']) * 100
# Dropping exact duplicate rows, if any
before = len(brands)
brands = brands.drop_duplicates(subset=['product_id'])
after = len(brands)
print(f"Removed {before - after} duplicate rows")

300
Removed 1 duplicate rows


In [14]:
# EXPORTING THE CLEANED DATA 
brands.to_csv('../data/cleaned/nykaa_brands_clean.csv', index=False)
import sqlite3
conn = sqlite3.connect('../sql/nykaa.db')
brands.to_sql('brands', conn, if_exists='replace', index=False)

3662

In [15]:
# Using SQL Queries to make Dashboard
queries = {
    'discount_trap': "SELECT brand_name, AVG(discount_pct) as avg_discount, AVG(rating) as avg_rating, COUNT(*) as product_count FROM brands GROUP BY brand_name HAVING avg_discount > (SELECT avg(discount_pct) FROM brands) AND avg_rating < (SELECT avg(rating) FROM brands) ORDER BY avg_discount DESC LIMIT 10",
    'merchandising_audit': "SELECT tags, AVG(rating) as avg_rating, AVG(rating_count) as avg_rating_count, COUNT(*) as product_count FROM brands GROUP BY tags ORDER BY avg_rating_count DESC",
    'visibility_bias': "SELECT listing_page_no, avg(rating_count) as avg_rating_count, avg(rating) as avg_rating, count(*) as product_count FROM brands GROUP BY listing_page_no ORDER BY listing_page_no LIMIT 20",
    'inventory_risk': "SELECT product_title, brand_name, price, rating_count FROM brands WHERE price > (SELECT AVG(price) FROM brands) AND rating_count < 20 ORDER BY price DESC LIMIT 15",
    'price_segment': "SELECT CASE WHEN price < 300 THEN 'Budget' WHEN price BETWEEN 300 AND 800 THEN 'Mid-range' ELSE 'Premium' END AS price_segment, AVG(rating) as avg_rating, COUNT(*) as product_count FROM brands GROUP BY price_segment"
}

for name, q in queries.items():
    pd.read_sql(q, conn).to_csv(f'../dashboard/{name}.csv', index=False)
    print(f"Saved {name}.csv")

# Also export the full cleaned table
brands.to_csv('../dashboard/brands_full.csv', index=False)
print("Saved brands_full.csv")

Saved discount_trap.csv
Saved merchandising_audit.csv
Saved visibility_bias.csv
Saved inventory_risk.csv
Saved price_segment.csv
Saved brands_full.csv
